# K1 — Cross-family Rank Sweep on Gemma 4 9B (Prediction P2)

**Base model:** `google/gemma-4-9b` (configurable below) · **Sweep:** r ∈ {4, 16, 64, 128} · **Hardware:** Kaggle T4 (16 GB)

Replicates Findings F1–F3 from the paper on a different model family. P2 says the proximity-gap sign flip should land in the **r ∈ [16, 64]** window regardless of architecture.

**Runtime:** ~6 hours total (~90 min × 4 ranks). Split across two 12-hour Kaggle sessions if needed by setting `RANKS_TO_RUN` below. **Token required:** YES — see HF_TOKEN cell.

> If `google/gemma-4-9b` returns 404, edit `MODEL_ID` to your confirmed Gemma 4 ID. Likely candidates: `google/gemma-4-9b-base`, `google/gemma-4-9b-pt`, or fall back to `google/gemma-2-9b`.


https://www.kaggle.com/code/akankshanarula/gemma

In [ ]:
# === Install Gemma 4-compatible dependencies (one-shot, ~3 min) ===
# Gemma 4 uses model_type="gemma4"; install a build with Gemma4 classes.
# Do not use --no-deps here: Kaggle images often have incompatible leftovers.
!pip install -q -U \
    "git+https://github.com/huggingface/transformers.git" \
    "peft>=0.17.0" \
    "trl>=0.18.0" \
    "datasets>=3.0.2" \
    "bitsandbytes>=0.44.1" \
    "accelerate>=1.8.0" \
    "lm-eval>=0.4.4" \
    "sentencepiece>=0.2.0" \
    "protobuf>=3.20.3" \
    "huggingface_hub>=0.31.0" \
    "safetensors>=0.4.1"

# This notebook is text-only. Some Kaggle images ship a torchvision build whose
# CUDA version does not match torch, which can break imports through optional
# vision integrations. Remove it and clear partial imports.
!pip uninstall -y -q torchvision
import sys, importlib.metadata as im
for _name in list(sys.modules):
    if _name == "torchvision" or _name.startswith("torchvision."):
        del sys.modules[_name]
print("install done; torchvision removed for text-only Transformers/PEFT imports")
for _pkg in ["transformers", "tokenizers", "peft", "accelerate", "bitsandbytes"]:
    try:
        print(_pkg, im.version(_pkg))
    except Exception as _e:
        print(_pkg, "version check failed:", _e)

# Fail fast if the active Python process is still seeing an older Transformers.
try:
    from transformers import Gemma4Config, Gemma4ForConditionalGeneration
    print("Gemma4 support OK:", Gemma4ForConditionalGeneration.__name__)
except Exception as _e:
    raise RuntimeError(
        "This kernel still cannot import Gemma4 classes. Restart the Kaggle session, "
        "rerun this install cell first, then run the notebook from the top. "
        f"Original error: {type(_e).__name__}: {_e}"
    )

In [ ]:
import os, gc, json, time, math, warnings, traceback
from pathlib import Path

# Text-only run: keep Transformers from touching an incompatible torchvision
# install before any transformers/peft imports happen.
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def disable_torchvision_for_transformers():
    try:
        import transformers.utils.import_utils as _tf_import_utils
        _tf_import_utils._torchvision_available = False
        _tf_import_utils._torchvision_version = "unavailable"
    except Exception as e:
        print(f"torchvision guard warning: {type(e).__name__}: {e}")

disable_torchvision_for_transformers()

import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

# GPU sanity
assert torch.cuda.is_available(), "No GPU detected. Switch Kaggle accelerator to GPU T4 or P100."
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
torch.set_float32_matmul_precision("high")

OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
print("output dir:", OUT)
print("Transformers torchvision integration disabled for this text-only run.")

In [ ]:
# === K1 config ===
# Based on the Gemma 4 documentation provided:
# E4B stands for "Effective 4 Billion" (Total 8B with Per-Layer Embeddings)
MODEL_ID = "google/gemma-4-E4B-it"

# Gemma 4 supports higher ranks; 4-128 is a solid range for experimentation
RANKS_TO_RUN = [4, 16, 64, 128]
TRAIN_STEPS = 500
N_TRAIN = 4096

# Gemma 4 Sampling Best Practices (from documentation)
SAMPLING_CONFIG = {
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 64
}

print(f"MODEL: {MODEL_ID} | ranks: {RANKS_TO_RUN}")

# Medical MMLU subjects for per-subject analysis
MED_SUBJECTS = [
    "clinical_knowledge", "medical_genetics", "college_medicine",
    "anatomy", "professional_medicine", "virology",
    "nutrition", "human_aging", "human_sexuality"
]

In [ ]:
# === HF_TOKEN setup (REQUIRED for Gemma — gated model) ===
# Steps if this fails:
#   1. https://huggingface.co/google/gemma-2-9b   (or gemma-4-9b)
#      → click "Acknowledge license"
#   2. Create a fine-grained token with READ access to gated repos
#      https://huggingface.co/settings/tokens
#   3. In Kaggle: Add-ons → Secrets → Add HF_TOKEN
#   4. In this notebook: Settings → Secrets → enable HF_TOKEN
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        f"HF_TOKEN secret not found ({type(e).__name__}). "
        "Gemma is gated — see instructions in this cell."
    )

from huggingface_hub import login, HfApi
login(token=HF_TOKEN, add_to_git_credential=False)
# Validate access to the chosen model
try:
    HfApi().model_info(MODEL_ID, token=HF_TOKEN)
    print(f"HF login OK; access to {MODEL_ID} confirmed.")
except Exception as e:
    raise RuntimeError(
        f"Cannot access {MODEL_ID}: {e}\n"
        f"Visit https://huggingface.co/{MODEL_ID} and click 'Acknowledge license' "
        "with the same HF account that owns your token."
    )

In [ ]:
# === Memory-safe quantisation: NF4 + double-quant + bf16 compute ===
from transformers import BitsAndBytesConfig
BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
print("BNB config ready (NF4 + double-quant + bf16 compute).")

In [ ]:
# === Load base model + tokenizer (cached after first run) ===
from transformers import AutoTokenizer, Gemma4ForConditionalGeneration
_Gemma4AutoModel = Gemma4ForConditionalGeneration
_MODEL_AUTO_CLASS = "Gemma4ForConditionalGeneration"

def load_base():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    model = _Gemma4AutoModel.from_pretrained(
        MODEL_ID,
        quantization_config=BNB,
        device_map={"": 0},
        dtype=torch.bfloat16,
        trust_remote_code=True,
        token=HF_TOKEN,
        attn_implementation="eager",  # safer on T4 than flash-attn
    )
    model.config.use_cache = False
    if hasattr(model.config, "text_config"):
        model.config.text_config.use_cache = False
    print(f"Loaded {MODEL_ID} with {_MODEL_AUTO_CLASS}")
    return model, tok

print("load_base() defined.")

In [ ]:
# === LoRA attachment helper ===
# Keep this cell safe even after a failed/partial torchvision import.
if "disable_torchvision_for_transformers" not in globals():
    import os, sys
    os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
    for _name in list(sys.modules):
        if _name == "torchvision" or _name.startswith("torchvision."):
            del sys.modules[_name]
    def disable_torchvision_for_transformers():
        try:
            import transformers.utils.import_utils as _tf_import_utils
            _tf_import_utils._torchvision_available = False
            _tf_import_utils._torchvision_version = "unavailable"
        except Exception as e:
            print(f"torchvision guard warning: {type(e).__name__}: {e}")

disable_torchvision_for_transformers()
from peft import LoraConfig, get_peft_model

def prepare_kbit_no_fp32_cast(model):
    """PEFT's helper casts non-4bit bf16/fp16 params to fp32; on Gemma E4B that OOMs."""
    for param in model.parameters():
        param.requires_grad = False
    if hasattr(model, "config"):
        model.config.use_cache = False
        if hasattr(model.config, "text_config"):
            model.config.text_config.use_cache = False
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    else:
        def _make_inputs_require_grad(module, inputs, output):
            output.requires_grad_(True)
        model.get_input_embeddings().register_forward_hook(_make_inputs_require_grad)
    if hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
    return model

def attach_lora(model, r=16, alpha=None, dropout=0.05):
    alpha = alpha or r
    model = prepare_kbit_no_fp32_cast(model)
    # Gemma 4 wraps quantized projections in Gemma4ClippableLinear; target the inner Linear4bit.
    cfg = LoraConfig(
        r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none",
        target_modules=[
            "q_proj.linear", "k_proj.linear", "v_proj.linear", "o_proj.linear",
            "gate_proj.linear", "up_proj.linear", "down_proj.linear",
        ],
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

print("attach_lora() defined: no PEFT fp32 cast path")

In [ ]:
# === MedQA loader (correct HF path, no loading script) ===
from datasets import load_dataset

def load_medqa(n_train=4096, n_test=200, seed=42):
    ds = load_dataset("GBaker/MedQA-USMLE-4-options-hf", token=HF_TOKEN)
    train = ds["train"].shuffle(seed=seed).select(range(min(n_train, len(ds["train"]))))
    test  = ds["test"].select(range(min(n_test, len(ds["test"]))))
    return train, test

def format_medqa_train(ex, tok, max_len=512):
    q = ex["sent1"]
    opts = "\n".join([f"{c}. {ex[k]}" for c, k in zip("ABCD", ["ending0","ending1","ending2","ending3"])])
    ans = "ABCD"[int(ex["label"])]
    text = f"Question: {q}\nOptions:\n{opts}\nAnswer: {ans}"
    enc = tok(text, truncation=True, max_length=max_len, padding="max_length", return_tensors="pt")
    enc = {k: v[0] for k, v in enc.items()}
    enc["labels"] = enc["input_ids"].clone()
    enc["labels"][enc["attention_mask"]==0] = -100
    return enc

print("MedQA loaders defined.")

In [ ]:
# === Fine-tune wrapper (LoRA, gradient checkpointing, OOM-safe defaults) ===
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset

class TokDS(Dataset):
    def __init__(self, hf_ds, tok, max_len=512):
        self.hf_ds, self.tok, self.max_len = hf_ds, tok, max_len
    def __len__(self): return len(self.hf_ds)
    def __getitem__(self, i):
        return format_medqa_train(self.hf_ds[i], self.tok, self.max_len)

def run_finetune(model, tok, train_ds, replay_ds=None, steps=500, r=16, lr=2e-4, run_name="exp"):
    model = attach_lora(model, r=r)
    if replay_ds is not None:
        from datasets import concatenate_datasets
        # 1:1 mix: interleave by repeating shorter
        combined = concatenate_datasets([train_ds, replay_ds])
        combined = combined.shuffle(seed=42)
    else:
        combined = train_ds
    train_torch = TokDS(combined, tok)
    args = TrainingArguments(
        output_dir=f"/kaggle/working/{run_name}",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        max_steps=steps,
        learning_rate=lr,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=25,
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        report_to="none",
        seed=42,
        dataloader_num_workers=2,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_torch)
    trainer.train()
    return model

print("run_finetune() defined.")

In [ ]:
# === MMLU per-subject evaluator via lm-eval-harness ===
import subprocess, json, tempfile

MMLU_SUBJECTS = [
 'abstract_algebra','anatomy','astronomy','business_ethics','clinical_knowledge',
 'college_biology','college_chemistry','college_computer_science','college_mathematics',
 'college_medicine','college_physics','computer_security','conceptual_physics',
 'econometrics','electrical_engineering','elementary_mathematics','formal_logic',
 'global_facts','high_school_biology','high_school_chemistry','high_school_computer_science',
 'high_school_european_history','high_school_geography','high_school_government_and_politics',
 'high_school_macroeconomics','high_school_mathematics','high_school_microeconomics',
 'high_school_physics','high_school_psychology','high_school_statistics','high_school_us_history',
 'high_school_world_history','human_aging','human_sexuality','international_law','jurisprudence',
 'logical_fallacies','machine_learning','management','marketing','medical_genetics','miscellaneous',
 'moral_disputes','moral_scenarios','nutrition','philosophy','prehistory','professional_accounting',
 'professional_law','professional_medicine','professional_psychology','public_relations',
 'security_studies','sociology','us_foreign_policy','virology','world_religions'
]

@torch.no_grad()
def eval_mmlu_subject(model, tok, subject, n_examples=50):
    """Lightweight in-notebook MMLU eval to avoid lm-eval's heavy harness."""
    from datasets import load_dataset
    try:
        ds = load_dataset("cais/mmlu", subject, split=f"test[:{n_examples}]", token=HF_TOKEN)
    except Exception as e:
        print(f"  skip {subject}: {e}")
        return None
    abcd_ids = [tok.encode(L, add_special_tokens=False)[-1] for L in [" A"," B"," C"," D"]]
    correct = 0
    for ex in ds:
        q = ex["question"]
        opts = "\n".join([f"{c}. {o}" for c, o in zip("ABCD", ex["choices"])])
        prompt = f"Question: {q}\nOptions:\n{opts}\nAnswer:"
        enc = tok(prompt, return_tensors="pt", truncation=True, max_length=480).to(model.device)
        out = model(**enc).logits[0, -1, :]
        pred = int(np.argmax([out[i].item() for i in abcd_ids]))
        if pred == ex["answer"]:
            correct += 1
    return correct / len(ds)

@torch.no_grad()
def eval_all_mmlu(model, tok, n_examples=50):
    rows = []
    for s in MMLU_SUBJECTS:
        acc = eval_mmlu_subject(model, tok, s, n_examples=n_examples)
        if acc is not None:
            rows.append({"subject": s, "accuracy": acc, "is_medical": s in MED_SUBJECTS})
    return pd.DataFrame(rows)

print("MMLU eval defined.")

In [ ]:
# === Load MedQA training data once ===
train_ds, _ = load_medqa(n_train=N_TRAIN, n_test=10)
print(f"MedQA train rows: {len(train_ds)}")

In [ ]:
# === Rank sweep ===
all_results = []
for r in RANKS_TO_RUN:
    print(f"\n{'='*60}\n=== K1 rank {r} ===\n{'='*60}")
    t0 = time.time()
    gc.collect(); torch.cuda.empty_cache()
    model, tok = load_base()
    model = run_finetune(model, tok, train_ds, steps=TRAIN_STEPS, r=r, run_name=f"K1_r{r}")
    ft_df = eval_all_mmlu(model, tok, n_examples=50)
    # Compute per-subject forgetting against base
    merged = ft_df.merge(base_df, on="subject", suffixes=("_ft","_base"))
    merged["forgetting"] = merged["accuracy_base"] - merged["accuracy_ft"]
    merged["lora_rank"] = r
    merged["is_medical"] = merged["is_medical_ft"]
    out = merged[["subject","lora_rank","is_medical","accuracy_base","accuracy_ft","forgetting"]]
    out.to_csv(OUT/f"k1_rank{r}.csv", index=False)
    all_results.append(out)
    print(f"  rank {r}  mean f̄ = {out['forgetting'].mean():.4f}  "
          f"med Δ = {out[out.is_medical]['forgetting'].mean() - out[~out.is_medical]['forgetting'].mean():+.4f}  "
          f"({(time.time()-t0)/60:.1f} min)")
    del model, tok; gc.collect(); torch.cuda.empty_cache()

if all_results:
    combined = pd.concat(all_results)
    combined.to_csv(OUT/"k1_all_ranks.csv", index=False)

In [ ]:
# === P2 falsification check ===
if all_results:
    summary = []
    for r in RANKS_TO_RUN:
        sub = combined[combined.lora_rank==r]
        med = sub[sub.is_medical]["forgetting"].mean()
        nmed = sub[~sub.is_medical]["forgetting"].mean()
        summary.append({"rank": r, "mean_f": sub["forgetting"].mean(),
                        "med_f": med, "nonmed_f": nmed, "delta": med-nmed})
    s = pd.DataFrame(summary)
    s.to_csv(OUT/"k1_summary.csv", index=False)
    print(s.to_string(index=False))
    print("\nP2 verdict:")
    print("  Sign-flip rank window = [where Δ goes from negative to positive]")
    deltas = s.set_index("rank")["delta"].to_dict()
    flipped = [r for r in sorted(deltas) if deltas[r] > 0]
    if flipped:
        first_pos = flipped[0]
        print(f"  → First rank with positive Δ on Gemma 4 9B: r = {first_pos}")
        if 16 <= first_pos <= 64:
            print(f"  → P2 CONFIRMED: critical rank within [16, 64] window.")
        else:
            print(f"  → P2 REFUTED: critical rank outside [16, 64] window.")
    else:
        print(f"  → No positive Δ at any rank tested. P2 indeterminate; try larger ranks.")